In [ ]:
library(GenomicRanges)
library(GenomicFeatures)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(org.Hs.eg.db)
library(AnnotationDbi)

outdir_da <- "/path/to/nature_study/ATAC_donor_level/donor_aware_DA"

dar <- read.delim(
  file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_edgeR_DA.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

head(dar)

In [ ]:
# check expected columns
stopifnot(all(c("peak_id", "seqnames", "start", "end", "logFC", "FDR") %in% colnames(dar)))

# peak GRanges
peaks_gr <- GRanges(
  seqnames = dar$seqnames,
  ranges = IRanges(start = dar$start, end = dar$end),
  peak_id = dar$peak_id
)

# hg38 gene models
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

# gene bodies
genes_gr <- genes(txdb)

# 1-bp TSS positions
tss_gr <- promoters(genes_gr, upstream = 0, downstream = 1)

# nearest TSS for each peak
hits <- distanceToNearest(peaks_gr, tss_gr, ignore.strand = FALSE)
hits_df <- as.data.frame(hits)

# map nearest gene IDs
nearest_gene_ids <- as.character(mcols(tss_gr)$gene_id[hits_df$subjectHits])

nearest_symbols <- mapIds(
  org.Hs.eg.db,
  keys = nearest_gene_ids,
  keytype = "ENTREZID",
  column = "SYMBOL",
  multiVals = "first"
)

nearest_genenames <- mapIds(
  org.Hs.eg.db,
  keys = nearest_gene_ids,
  keytype = "ENTREZID",
  column = "GENENAME",
  multiVals = "first"
)

annot_df <- data.frame(
  peak_id = dar$peak_id[hits_df$queryHits],
  nearest_gene_id = nearest_gene_ids,
  SYMBOL = unname(nearest_symbols),
  GENENAME = unname(nearest_genenames),
  distanceToTSS = hits_df$distance,
  stringsAsFactors = FALSE
)

# join back to DAR table
dar_annot <- merge(dar, annot_df, by = "peak_id", all.x = TRUE, sort = FALSE)

# restore original row order
dar_annot <- dar_annot[match(dar$peak_id, dar_annot$peak_id), ]

# direction label
dar_annot$direction <- ifelse(
  dar_annot$logFC > 0, "higher_in_Ts21",
  ifelse(dar_annot$logFC < 0, "higher_in_disomy", "no_change")
)

# save
write.table(
  dar_annot,
  file = file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_edgeR_DA_nearestTSS.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

head(dar_annot[, c("peak_id", "seqnames", "start", "end", "SYMBOL", "distanceToTSS", "logFC", "FDR")], 20)

In [ ]:
genes_b <- c(
  "EBF1", "PAX5", "CD24", "DNTT",
  "BCL11A", "SPI1", "SPIB", "IRF8",
  "TCF3", "TCF12", "IKZF1", "RAG1",
  "BLNK", "IGLL1", "BACH2", "CD79A", "CD79B"
)

b_hits <- dar_annot[dar_annot$SYMBOL %in% genes_b, ]
b_hits <- b_hits[order(b_hits$FDR, abs(b_hits$logFC), decreasing = c(FALSE, TRUE)), ]

write.table(
  b_hits,
  file = file.path(outdir_da, "HSC_MPP_DAR_Blineage_nearestTSS_hits.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

b_hits[, c("peak_id", "seqnames", "start", "end", "SYMBOL", "distanceToTSS", "logFC", "FDR")]

In [ ]:
sig_b_hits <- b_hits[b_hits$FDR <= 0.05, ]
sig_b_hits[order(sig_b_hits$FDR), c("peak_id", "seqnames", "start", "end", "SYMBOL", "distanceToTSS", "logFC", "FDR")]

In [ ]:
top_b_hits_relaxed <- b_hits[b_hits$FDR <= 0.2, ]
top_b_hits_relaxed[order(top_b_hits_relaxed$FDR), c("peak_id", "seqnames", "start", "end", "SYMBOL", "distanceToTSS", "logFC", "FDR")]

In [ ]:
outdir_da <- "/path/to/nature_study/ATAC_donor_level/donor_aware_DA"

da_hsc_tab <- read.delim(
  file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_edgeR_DA.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

head(da_hsc_tab)

# export the full HSC_MPP DAR table
write.table(
  da_hsc_tab,
  file = file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_edgeR_DA.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

saveRDS(
  da_hsc_tab,
  file = file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_edgeR_DA.rds")
)

# optional: export a tiny summary table too
dar_summary <- data.frame(
  metric = c("tested", "significant_FDR_0.05", "higher_in_disomy", "higher_in_Ts21"),
  value = c(171322, 1667, 988, 679)
)

write.table(
  dar_summary,
  file = file.path(outdir_da, "HSC_MPP_Ts21_vs_disomy_DAR_summary.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

dar_summary